# FraudGuard: AI-Driven Credit Card Fraud Detection for Banking

**Author:** [Your Name]  
**Date:** 2026-06-18  
**Objective:** Build an end-to-end, production-ready machine learning pipeline for credit card fraud detection, targeting entry-level AI/ML roles in the banking sector.

---
## Executive Summary
Credit card fraud is a multi-billion dollar problem for financial institutions. A key challenge is the extreme class imbalance of fraudulent transactions. This project demonstrates a rigorous data science workflow: from exploratory data analysis (EDA) and advanced feature engineering, to model benchmarking with cost-sensitive threshold tuning and explainability (XAI).

**Key Business Metrics:**
- **False Negative Cost:** Missing a fraud case can cost the bank upwards of 100 USD per transaction.
- **False Positive Cost:** Incorrectly blocking a legitimate transaction damages customer trust.
- **Optimization Goal:** Minimize total cost, not just maximize accuracy.

**Dataset:** ULB Credit Card Fraud Detection (284,807 transactions, 492 frauds, Imbalance Ratio ~577:1).

In [ ]:
# Cell 1: Environment Setup & Imports
import sys
import os
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Ensure src is in path
sys.path.insert(0, os.path.abspath('../src'))
from data import load_real_creditcard

# Settings
sns.set_theme(style='whitegrid')
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Environment ready.')

In [ ]:
# Cell 2: Data Loading & Initial Inspection
data = load_real_creditcard()
df = pd.DataFrame(data['X'], columns=data['features'])
df['Class'] = data['y']

print(f'Dataset Shape: {df.shape}')
print(f'Fraud Rate: {df["Class"].mean():.4%}')
print(f'\nFirst 5 rows:\n{df.head()}')

## 1. Exploratory Data Analysis (EDA)
Understanding the data is crucial before modeling. We analyze the class imbalance, feature correlations, and the temporal nature of fraud.

In [ ]:
# Cell 3: Class Imbalance & Time Analysis
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Class Distribution
class_counts = df['Class'].value_counts() 
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette=['#3498db', '#e74c3c'])
axes[0].set_xticklabels(['Normal (0)', 'Fraud (1)'])
axes[0].set_title('Transaction Class Distribution')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 500, str(v), ha='center', va='bottom')

# 2. Fraud Rate by Hour
df['Hour'] = (df['Time'] // 3600) % 24
fraud_by_hour = df.groupby('Hour')['Class'].mean()
sns.barplot(x=fraud_by_hour.index, y=fraud_by_hour.values, ax=axes[1], color='#e74c3c')
axes[1].set_title('Fraud Rate by Hour of Day')
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_ylabel('Fraud Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 4: Transaction Amount Analysis
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# 1. Amount Distribution (Log Scale)
sns.histplot(df[df['Class']==0]['Amount'], bins=50, ax=axes[0], label='Normal', color='#3498db', log_scale=True, stat='density', alpha=0.6)
sns.histplot(df[df['Class']==1]['Amount'], bins=50, ax=axes[0], label='Fraud', color='#e74c3c', log_scale=True, stat='density', alpha=0.6)
axes[0].set_title('Transaction Amount Distribution (Log Scale)')
axes[0].set_xlabel('Amount (USD)')
axes[0].legend()

# 2. Box Plot of Key Features
sample_df = df.sample(n=20000, random_state=42)
sample_df_melted = pd.melt(sample_df, id_vars=['Class'], value_vars=['V14', 'V4', 'V12', 'V10'])
sns.boxplot(x='variable', y='value', hue='Class', data=sample_df_melted, ax=axes[1], palette=['#3498db', '#e74c3c'])
axes[1].set_title('Distribution of Key PCA Components by Class')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Correlation Analysis
plt.figure(figsize=(20, 15))
corr = df.corr()
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix')
plt.show()

## 2. Advanced Feature Engineering
Raw features are rarely sufficient. We derive new features that capture behavioral patterns and non-linear relationships.

In [ ]:
# Cell 6: Feature Engineering Pipeline

# 1. Log-transform Amount to handle skewness
df['log_amount'] = np.log1p(df['Amount'] + 1e-6)

# 2. Time-based features
df['hour'] = (df['Time'] // 3600) % 24
df['day_segment'] = pd.cut(df['hour'], bins=[0, 6, 12, 18, 24], labels=['Night', 'Morning', 'Afternoon', 'Evening'])

# 3. Amount-based interactions (identifies unusual spending for a given PCA footprint)
df['V14_x_Amount'] = df['V14'] * df['Amount']
df['V4_x_Amount'] = df['V4'] * df['Amount']
df['V12_x_Amount'] = df['V12'] * df['Amount']

# 4. Velocity-like features (simulated for this dataset)
# Note: In a real system, we would count transactions per card/hour from a database
df['amount_deviation'] = df['Amount'] - df.groupby('hour')['Amount'].transform('median')

feature_cols = [c for c in df.columns if c not in ['Class', 'Time', 'day_segment']]
X = df[feature_cols]
y = df['Class']

print(f'Total Features after Engineering: {len(feature_cols)}')
print(f"New features added: { [c for c in feature_cols if c not in data['features']] }")

## 3. Modeling: Benchmarking & Imbalance Handling
We establish a baseline with Logistic Regression, then benchmark against tree-based ensembles (LightGBM) which natively handle non-linearities. We use class_weight='balanced' to mitigate the class imbalance.

**Why Stratified Split?** In imbalanced data, a random split might place all fraud cases in one fold. Stratification preserves the class distribution across train and test sets.

In [ ]:
# Cell 7: Model Training & Evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, roc_auc_score, f1_score, 
                             precision_score, recall_score, confusion_matrix, 
                             classification_report)

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

# Standard Scaling
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Define Models
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=150, random_state=42, n_jobs=-1)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_s, y_train)
    proba = model.predict_proba(X_test_s)[:, 1]
    pred = model.predict(X_test_s)
    
    results[name] = {
        'model': model,
        'proba': proba,
        'pred': pred,
        'pr_auc': average_precision_score(y_test, proba),
        'roc_auc': roc_auc_score(y_test, proba),
        'f1': f1_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred)
    }

# Add LightGBM if available
try:
    import lightgbm as lgb
    print('Training LightGBM...')
    lgb_model = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300, learning_rate=0.05, random_state=42)
    lgb_model.fit(X_train_s, y_train)
    lgb_proba = lgb_model.predict_proba(X_test_s)[:, 1]
    lgb_pred = lgb_model.predict(X_test_s)
    results['LightGBM'] = {
        'model': lgb_model,
        'proba': lgb_proba,
        'pred': lgb_pred,
        'pr_auc': average_precision_score(y_test, lgb_proba),
        'roc_auc': roc_auc_score(y_test, lgb_proba),
        'f1': f1_score(y_test, lgb_pred),
        'precision': precision_score(y_test, lgb_pred),
        'recall': recall_score(y_test, lgb_pred)
    }
except ImportError:
    print('LightGBM not installed, skipping.')

# Print Results
results_df = pd.DataFrame({k: {m: v[m] for m in ['pr_auc', 'roc_auc', 'f1', 'precision', 'recall']} for k, v in results.items()}).T
print('\n🏆 Model Performance Benchmark:')
print(results_df)

## 4. Cost-Sensitive Threshold Optimization
Optimizing for accuracy in fraud detection is dangerous. A model that predicts 'Normal' for everything gets 99.8% accuracy but catches 0% fraud. We must optimize for business cost.

**Assumed Costs (simulated):**
- **False Negative (FN):** Missing fraud = 100 USD
- **False Positive (FP):** Blocking a good transaction = 10 USD

In [ ]:
# Cell 8: Threshold Optimization based on Cost
fn_cost, fp_cost = 100, 10

best_model_name = results_df['pr_auc'].idxmax()
print(f'Best model by PR-AUC: {best_model_name}')

y_proba = results[best_model_name]['proba']
thresholds = np.linspace(0.01, 0.99, 100)
costs = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred_t)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0
    cost = (fn * fn_cost) + (fp * fp_cost)
    costs.append(cost)

optimal_idx = np.argmin(costs)
optimal_threshold = thresholds[optimal_idx]

plt.figure(figsize=(12, 6))
plt.plot(thresholds, costs, color='#e74c3c', linewidth=2)
plt.axvline(optimal_threshold, color='#2ecc71', linestyle='--', label=f'Optimal Threshold = {optimal_threshold:.3f}')
plt.xlabel('Decision Threshold')
plt.ylabel('Total Business Cost (USD)')
plt.title('Cost-Sensitive Threshold Optimization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f'Optimal Threshold: {optimal_threshold:.4f}')
print(f'Minimum Cost at Threshold: {costs[optimal_idx]:,.2f} USD')

## 5. Explainability (XAI) with SHAP
Banking regulators demand 'White-Box AI'. SHAP (SHapley Additive exPlanations) uses game theory to explain the marginal contribution of each feature to a prediction. This is critical for justifying why a transaction was blocked.

In [ ]:
# Cell 9: SHAP Analysis
try:
    import shap
    
    # Use TreeExplainer for tree-based models, KernelExplainer for linear
    if best_model_name == 'LightGBM':
        explainer = shap.TreeExplainer(results[best_model_name]['model'])
        shap_values = explainer.shap_values(X_test_s)
        
        # Summary Plot
        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values[1], X_test, feature_names=feature_cols, show=False, max_display=15)
        plt.title('SHAP Feature Importance Summary (Fraud Class)')
        plt.show()
    else:
        print('SHAP summary plot is most informative for tree-based models.')
        
except ImportError:
    print('SHAP library not installed. Run: pip install shap')

In [ ]:
# Cell 10: Save Best Model & Artifacts
import pickle
from pathlib import Path

# Save model, scaler, and optimal threshold
artifact = {
    'model': results[best_model_name]['model'],
    'scaler': scaler,
    'features': feature_cols,
    'threshold': float(optimal_threshold),
    'metrics': results[best_model_name],
    'name': best_model_name
}

model_path = Path('../models/master_model.pkl')
model_path.parent.mkdir(parents=True, exist_ok=True)
with open(model_path, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Best model ({best_model_name}) saved to {model_path}')

## Conclusion & Next Steps

This notebook demonstrates a rigorous, end-to-end workflow for fraud detection in banking. Key takeaways:
1. **Always stratify** when dealing with imbalanced data.
2. **Engineer features** that capture domain-specific logic (e.g., time of day, amount interactions).
3. **Optimize for business cost**, not just accuracy.
4. **Explainability is non-negotiable** in regulated industries.

**Next Steps for Production:**
- Implement real-time streaming inference using Apache Kafka + FastAPI.
- Set up model monitoring (drift detection) with Evidently AI.
- Explore Graph Neural Networks (GNNs) to model transaction networks.
- Deploy via Docker and orchestrate with Kubernetes.